In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# Load MobileNet
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(train_data, validation_data=val_data, epochs=10)

model.save("rice_disease_model.h5")

Found 168 images belonging to 4 classes.
Found 41 images belonging to 4 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 3s/step - accuracy: 0.6310 - loss: 0.9227 - val_accuracy: 0.8049 - val_loss: 0.4613
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.9226 - loss: 0.2147 - val_accuracy: 0.9268 - val_loss: 0.2818
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.9643 - loss: 0.1113 - val_accuracy: 0.8049 - val_loss: 0.4409
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.9762 - loss: 0.0826 - val_accuracy: 0.9268 - val_loss: 0.1987
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.9821 - loss: 0.0515 - val_accuracy: 0.9512 - val_loss: 0.2484
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.9881 - loss: 0.0307 - val_accuracy: 0.8293 - val_loss: 0.2733
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.9940 - loss: 0.0248 - val_accuracy: 0.8293 - val_loss: 0.3081
Ep

In [2]:
print(train_data.class_indices)

{'bacterial_blight': 0, 'brown_spot': 1, 'healthy': 2, 'leaf_smut': 3}


In [4]:
disease_info = {
    "Brown_Spot": {
        "description": "Fungal disease causing brown lesions.",
        "remedy": "Use fungicides like Mancozeb and maintain proper drainage."
    },
    "Bacterial_Blight": {
        "description": "Bacterial infection causing yellowing.",
        "remedy": "Use resistant varieties and avoid excess nitrogen."
    },
    "Leaf_Smut": {
        "description": "Black powdery lesions.",
        "remedy": "Seed treatment and proper field sanitation."
    },
    "Healthy": {
        "description": "No disease detected.",
        "remedy": "Maintain regular monitoring."
    }
}